# imports 

In [0]:
from pyspark.sql.functions import *
from pyspark.sql.types import StringType,DateType

# read date from Bronze Layer

In [0]:
df = spark.read.table('`e-commerce-databricks-project`.bronze_layer.erp_px_cat_g1v2')

# apply Data Transformations for silver Table

In [0]:
df.display()

ID,CAT,SUBCAT,MAINTENANCE
AC_BR,Accessories,Bike Racks,Yes
AC_BS,Accessories,Bike Stands,No
AC_BC,Accessories,Bottles and Cages,No
AC_CL,Accessories,Cleaners,Yes
AC_FE,Accessories,Fenders,No
AC_HE,Accessories,Helmets,Yes
AC_HP,Accessories,Hydration Packs,No
AC_LI,Accessories,Lights,Yes
AC_LO,Accessories,Locks,Yes
AC_PA,Accessories,Panniers,No


## Renaming columns

In [0]:

RENAME_MAP = {
    "id": "category_id",
    "cat": "category",
    "subcat": "subcategory",
    "maintenance": "maintenance_flag"
}
for old,new in RENAME_MAP.items():
    df = df.withColumnRenamed(old,new)

## Triming String columns

In [0]:
for colName, colType in df.dtypes:
    if colType == 'string':
        df = df.withColumn(colName,trim(col(colName)))
df.display()

category_id,category,subcategory,maintenance_flag
AC_BR,Accessories,Bike Racks,Yes
AC_BS,Accessories,Bike Stands,No
AC_BC,Accessories,Bottles and Cages,No
AC_CL,Accessories,Cleaners,Yes
AC_FE,Accessories,Fenders,No
AC_HE,Accessories,Helmets,Yes
AC_HP,Accessories,Hydration Packs,No
AC_LI,Accessories,Lights,Yes
AC_LO,Accessories,Locks,Yes
AC_PA,Accessories,Panniers,No


## Normalize Maintenance Flag to be Boolean

In [0]:
df = df.withColumn(
    "maintenance_flag",
       when(upper(col("maintenance_flag")) == "YES", lit(True))
      .when(upper(col("maintenance_flag")) == "NO", lit(False))
      .otherwise(None)
)

## Final Look

In [0]:
df.display()

category_id,category,subcategory,maintenance_flag
AC_BR,Accessories,Bike Racks,true
AC_BS,Accessories,Bike Stands,false
AC_BC,Accessories,Bottles and Cages,false
AC_CL,Accessories,Cleaners,true
AC_FE,Accessories,Fenders,false
AC_HE,Accessories,Helmets,true
AC_HP,Accessories,Hydration Packs,false
AC_LI,Accessories,Lights,true
AC_LO,Accessories,Locks,true
AC_PA,Accessories,Panniers,false


# write data into Silver Layer

In [0]:
df.write.mode('overwrite').format('delta').saveAsTable('`e-commerce-databricks-project`.silver_layer.erp_categories_details')